# iMD Agent

Example of defining a custom iMD Agent that dynamically updates interactions based on incoming frame updates.

## Setup runner

In [ ]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../ase/openmm_files/helicene.xml")
imd_runner = OmniRunner.with_basic_server(simulation)
imd_runner.load(0)

## Setup viewer

In [ ]:
from nanover.jupyter import NGLClient

client = NGLClient.from_runner(imd_runner)
client.view

## Define agent

In [ ]:
import numpy as np
import math

from nanover.trajectory import FrameData
from nanover.imd import ParticleInteraction
from nanover.jupyter import ImdAgent


class SqueezerAgent(ImdAgent):
    strength = 1
    count = 0

    def update_interactions(self, full_frame: FrameData, frame_update: FrameData):
        # oscillate scale between 0 and 1
        self.count += 1
        time = self.count / 30  # 30fps
        u = (time / 4) % 1
        scale = math.sin(u * math.pi) * 10

        # scale between crushing particles to point and pushing them away
        centroid = np.average(full_frame.particle_positions, axis=0)
        vecs = full_frame.particle_positions - centroid
        norm = np.linalg.norm(vecs, axis=1).reshape(-1, 1)
        unit = vecs / norm
        next = full_frame.particle_positions + unit * scale

        # update interactions
        for i in range(full_frame.particle_count):
            interaction = ParticleInteraction(position=[int(x) for x in next[i]], particles=[i], interaction_type="spring", scale=self.strength, max_force=100)
            self.update_interaction(f"squeeze.{i}", interaction)

## Attach and start agent

In [ ]:
agent = SqueezerAgent.from_runner(imd_runner)
agent.start()